In [4]:
import os
import sys
import urllib.request
from io import BytesIO

from astroML.datasets import fetch_dr7_quasar
import astroML.datasets.dr7_quasar as dr7
from astroML.datasets.tools.download import bytes_to_string

# Patch AstroML download to handle servers that omit Content-Length header.
# This prevents KeyError during download.

def _download_with_progress_bar_fixed(data_url, return_buffer=False):
    num_units = 40
    fhandle = urllib.request.urlopen(data_url)

    info = dict(fhandle.info())
    content_length = info.get('Content-Length')
    if content_length is not None:
        content_length = int(content_length.strip())
        chunk_size = max(1, content_length // num_units)
    else:
        # fallback to a sensible chunk size when total size is unknown
        chunk_size = 1024 * 1024
        content_length = None

    print("Downloading %s" % data_url)
    nchunks = 0
    buf = BytesIO()

    while True:
        next_chunk = fhandle.read(chunk_size)
        nchunks += 1
        if next_chunk:
            buf.write(next_chunk)
            if content_length is not None:
                s = ('[' + nchunks * '='
                     + (num_units - 1 - nchunks) * ' '
                     + ']  {} / {}   \r'.format(bytes_to_string(buf.tell()),
                                                bytes_to_string(content_length)))
                sys.stdout.write(s)
                sys.stdout.flush()
        else:
            if content_length is not None:
                sys.stdout.write('\n')
            break

    buf.seek(0)
    if return_buffer:
        return buf
    else:
        return buf.getvalue()

# Apply the patch to the module that fetch_dr7_quasar uses.
dr7.download_with_progress_bar = _download_with_progress_bar_fixed

# Remove any partially downloaded file so fetch_dr7_quasar will re-download.
# The cache directory is typically '~/astroML_data'.
if os.path.exists(os.path.expanduser('~/astroML_data/dr7qso.npy')):
    os.remove(os.path.expanduser('~/astroML_data/dr7qso.npy'))

# Fetch the quasar data
# This should now succeed even if the remote server omits Content-Length.
data = fetch_dr7_quasar()

# select the first 10000 points
data = data[:10000]

z = data['redshift']

downloading DR7 quasar dataset from http://das.sdss.org/va/qsocat/dr7qso.dat.gz to /home/matti/astroML_data


In [ ]:
from urllib.request import urlretrieve
urlretrieve('https:')